## Chargement initiaux

In [ ]:
# Bibliothèques utilisées pour cette première étape
# ------------------------------------------------------------
import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer
from torch.utils.data import Dataset, DataLoader
from transformers import DataCollatorWithPadding
from transformers import AutoModelForSequenceClassification
import time
from tqdm.auto import tqdm
from sklearn.metrics import accuracy_score, f1_score

# Configuration du matériel
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Calcul sur :", device)
if device.type == "cuda":
    print("GPU :", torch.cuda.get_device_name(0))

# Chargement du dataset préparé
# ------------------------------------------------------------
df = pd.read_csv("../csv/split_text_data.csv")

# Récupération des ensembles existants
# ------------------------------------------------------------
train_df = df[df["split"] == "train"].copy()
val_df = df[df["split"] == "val"].copy()
test_df = df[df["split"] == "test"].copy()

X_train_text = train_df["texte_traduit"]
X_val_text = val_df["texte_traduit"]
X_test_text = test_df["texte_traduit"]
print("Train :", len(train_df))
print("Validation :", len(val_df))
print("Test :", len(test_df))
print("Nombre de classes :", df["prdtypecode"].nunique())

# Encodage des 27 classes en indices de 0 à 26
# ------------------------------------------------------------
classes = sorted(train_df["prdtypecode"].unique())

code_to_idx = {code: i for i, code in enumerate(classes)}
idx_to_code = {i: code for code, i in code_to_idx.items()}

y_train = train_df["prdtypecode"].map(code_to_idx).tolist()
y_val = val_df["prdtypecode"].map(code_to_idx).tolist()

print("Correspondance indice → prdtypecode :")
print(idx_to_code)

Calcul sur : cuda
GPU : Quadro RTX 5000
Train : 66802
Validation : 8351
Test : 8350
Nombre de classes : 27
Correspondance indice → prdtypecode :
{0: np.int64(10), 1: np.int64(40), 2: np.int64(50), 3: np.int64(60), 4: np.int64(1140), 5: np.int64(1160), 6: np.int64(1180), 7: np.int64(1280), 8: np.int64(1281), 9: np.int64(1300), 10: np.int64(1301), 11: np.int64(1302), 12: np.int64(1320), 13: np.int64(1560), 14: np.int64(1920), 15: np.int64(1940), 16: np.int64(2060), 17: np.int64(2220), 18: np.int64(2280), 19: np.int64(2403), 20: np.int64(2462), 21: np.int64(2522), 22: np.int64(2582), 23: np.int64(2583), 24: np.int64(2585), 25: np.int64(2705), 26: np.int64(2905)}


## Charger le tokenizer
on va utiliser le tokenizer et le vocabulaire associés à CamemBERT : ils correspondent à ceux utilisés lors de son préentraînement.

In [ ]:
# Chargement du tokenizer associé à CamemBERT
# ------------------------------------------------------------
nom_modele = "almanach/camembert-base"

tokenizer = AutoTokenizer.from_pretrained(nom_modele)

print("Taille du vocabulaire :", len(tokenizer))

c:\Users\antoi\Documents\GitHub\sep25_alt2_iia_rakuten_2\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\antoi\Documents\GitHub\sep25_alt2_iia_rakuten_2\venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\antoi\.cache\huggingface\hub\models--almanach--camembert-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run

Taille du vocabulaire : 32005


### Exemple sur une fiche produit

In [6]:
# Exemple de tokenisation
# ------------------------------------------------------------
texte_exemple = X_train_text.iloc[0]

tokens = tokenizer.tokenize(texte_exemple)
encodage = tokenizer(texte_exemple)

print("Texte :", texte_exemple)
print("\nTokens :", tokens)
print("\nIdentifiants :", encodage["input_ids"])
print("\nMasque d'attention :", encodage["attention_mask"])

Texte : Olivia: Carnet personnalisé / 150 pages / Grille de points / Ca Din A5 / Design de roses

Tokens : ['▁Olivia', ':', '▁Carnet', '▁personnalisé', '▁/', '▁150', '▁pages', '▁/', '▁Gr', 'ille', '▁de', '▁points', '▁/', '▁Ca', '▁Din', '▁A', '5', '▁/', '▁Design', '▁de', '▁roses']

Identifiants : [5, 29694, 92, 25146, 5666, 268, 2802, 1256, 268, 4552, 2068, 8, 725, 268, 1043, 14301, 114, 800, 268, 2934, 8, 9330, 6]

Masque d'attention : [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


## Mesurer la longueur des textes
On va regarder la distribution des longueurs sur le train uniquement. Cela nous aidera à choisir une longueur maximale en sachant quelle proportion de textes sera tronquée.

In [7]:
# Nombre de tokens par texte d'entraînement
# ------------------------------------------------------------
nb_tokens_speciaux = tokenizer.num_special_tokens_to_add()

longueurs_train = np.array([
    len(tokenizer.tokenize(texte)) + nb_tokens_speciaux
    for texte in X_train_text
])

print("Longueur moyenne :", round(longueurs_train.mean(), 1))
print("Longueur médiane :", np.median(longueurs_train))
print("Longueur maximale :", longueurs_train.max())

print("\nDistribution des longueurs :")
for percentile in [50, 75, 90, 95, 99]:
    valeur = np.percentile(longueurs_train, percentile)
    print(f"{percentile} % des textes ont au maximum {valeur:.0f} tokens")

print("\nProportion de textes qui seraient tronqués :")
for limite in [128, 256, 512]:
    proportion = (longueurs_train > limite).mean() * 100
    print(f"Limite {limite} : {proportion:.2f} %")

Token indices sequence length is longer than the specified maximum sequence length for this model (746 > 512). Running this sequence through the model will result in indexing errors


Longueur moyenne : 134.4
Longueur médiane : 77.0
Longueur maximale : 3263

Distribution des longueurs :
50 % des textes ont au maximum 77 tokens
75 % des textes ont au maximum 197 tokens
90 % des textes ont au maximum 317 tokens
95 % des textes ont au maximum 438 tokens
99 % des textes ont au maximum 748 tokens

Proportion de textes qui seraient tronqués :
Limite 128 : 38.10 %
Limite 256 : 14.82 %
Limite 512 : 2.96 %


## Tokeniser les ensembles
On applique le tokenizer à tous les textes du train et de la validation

In [ ]:
# Configuration
# ------------------------------------------------------------
MAX_LEN = 512
BATCH_SIZE = 8

# Tokenisation et troncature
# ------------------------------------------------------------
encodages_train = tokenizer(
    X_train_text.tolist(),
    truncation=True, # limite les séquences à MAX_LEN
    max_length=MAX_LEN
)

encodages_val = tokenizer(
    X_val_text.tolist(),
    truncation=True,
    max_length=MAX_LEN
)

print("Textes train encodés :", len(encodages_train["input_ids"]))
print("Textes validation encodés :", len(encodages_val["input_ids"]))

Textes train encodés : 66802
Textes validation encodés : 8351


## Création du Dataset
On crée une classe qui renvoie les données et la cible d’une fiche. Cette fois, chaque fiche est renvoyée sous forme de dictionnaire

In [ ]:
# Dataset pour CamemBERT
# ------------------------------------------------------------
class TexteDataset(Dataset):

    def __init__(self, encodages, labels):
        self.encodages = encodages
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            "input_ids": self.encodages["input_ids"][idx],
            "attention_mask": self.encodages["attention_mask"][idx],
            "labels": self.labels[idx]
        }


dataset_train = TexteDataset(encodages_train, y_train)
dataset_val = TexteDataset(encodages_val, y_val)

## Créer les DataLoader avec padding dynamique
Pour le modèle BiLSTM, on avait écrit une fonction **collate_fn** pour compléter les séquences jusqu’à la longueur maximale du batch. Ici Hugging Face nous fournit un outil qui fait ce travail pour son tokenizer : **DataCollatorWithPadding**


In [ ]:
# Padding dynamique
# ------------------------------------------------------------
collator = DataCollatorWithPadding(tokenizer=tokenizer)

# DataLoaders
# ------------------------------------------------------------
train_loader = DataLoader(
    dataset_train,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collator
)

val_loader = DataLoader(
    dataset_val,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collator
)

# Pour chaque batch, le collator ajoute les tokens de padding nécessaires, ajuste les masques d’attention et convertit les données en tenseurs PyTorch
# Les textes sont donc complétés jusqu’au plus long du batch, dans la limite de 512 tokens.

## Chargement du modèle CamemBERT

In [12]:
# Chargement du modèle préentraîné
# ------------------------------------------------------------
model = AutoModelForSequenceClassification.from_pretrained(
    nom_modele,
    num_labels=len(classes)
).to(device)

print("Modèle sur :", next(model.parameters()).device)
print("Nombre de paramètres :", sum(p.numel() for p in model.parameters()))

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Some weights of CamembertForSequenceClassification were not initialized from the model checkpoint at almanach/camembert-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Modèle sur : cuda:0
Nombre de paramètres : 110642715


## Test sur un lot de produit

In [ ]:
# Récupération d'un batch et transfert sur le GPU
# ------------------------------------------------------------
batch = next(iter(train_loader))
batch = {cle: valeur.to(device) for cle, valeur in batch.items()}

# Propagation avant, sans calcul des gradients
# ------------------------------------------------------------
model.eval()

with torch.no_grad():
    sorties = model(**batch)

print("Dimensions des textes :", batch["input_ids"].shape)
print("Dimensions des labels :", batch["labels"].shape) # Labels sont les bonnes classes pour chaque texte du batch
print("Dimensions des logits :", sorties.logits.shape) # logits sont des scores bruts pour chaque classe, avant application de softmax (conversion en probabilités)
# Donc pour une fiche, CamemBERT produit 27 scores, un pour chaque classe. Le score le plus élevé correspond à la classe prédite par le modèle.
print("Loss :", sorties.loss.item())

Dimensions des textes : torch.Size([8, 512])
Dimensions des labels : torch.Size([8])
Dimensions des logits : torch.Size([8, 27])
Loss : 3.2728993892669678


## Tester la mémoire avant de lancer l’entraînement

In [14]:
# Test d'une propagation avant et arrière
# ------------------------------------------------------------
model.train()
model.zero_grad()

torch.cuda.reset_peak_memory_stats()

sorties = model(**batch)
loss = sorties.loss
loss.backward()

print("Loss :", loss.item())
print(
    "Pic de mémoire GPU :",
    round(torch.cuda.max_memory_allocated() / 1024**3, 2),
    "Go"
)

# Libération des gradients après le test
model.zero_grad()

Loss : 3.2703964710235596
Pic de mémoire GPU : 2.87 Go


## Premier entrainement

In [16]:
# Paramètres du premier entraînement
# ------------------------------------------------------------
EPOCHS = 3
LEARNING_RATE = 2e-5

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)

historique_camembert = []
meilleur_f1 = -1

for epoch in range(1, EPOCHS + 1):
    debut = time.perf_counter()
    torch.cuda.reset_peak_memory_stats()

    # Entraînement
    # ------------------------------------------------------------
    model.train()
    loss_train = 0

    for batch in tqdm(train_loader, desc=f"Train {epoch}/{EPOCHS}"):
        batch = {cle: valeur.to(device) for cle, valeur in batch.items()}

        optimizer.zero_grad()

        sorties = model(**batch)
        loss = sorties.loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        loss_train += loss.item() * batch["labels"].size(0)

    loss_train /= len(dataset_train)

    # Validation
    # ------------------------------------------------------------
    model.eval()
    loss_val = 0
    predictions = []
    labels_reels = []

    with torch.no_grad():
        for batch in tqdm(val_loader, desc=f"Validation {epoch}/{EPOCHS}"):
            batch = {cle: valeur.to(device) for cle, valeur in batch.items()}

            sorties = model(**batch)
            loss_val += sorties.loss.item() * batch["labels"].size(0)

            predictions.extend(sorties.logits.argmax(dim=1).cpu().tolist())
            labels_reels.extend(batch["labels"].cpu().tolist())

    loss_val /= len(dataset_val)

    accuracy = accuracy_score(labels_reels, predictions)
    f1_macro = f1_score(labels_reels, predictions, average="macro")
    duree = time.perf_counter() - debut
    vram = torch.cuda.max_memory_allocated() / 1024**3

    historique_camembert.append({
        "epoch": epoch,
        "train_loss": loss_train,
        "val_loss": loss_val,
        "accuracy": accuracy,
        "f1_macro": f1_macro,
        "duree_s": duree,
        "vram_go": vram
    })

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"Train loss : {loss_train:.4f} | "
        f"Val loss : {loss_val:.4f} | "
        f"Accuracy : {accuracy:.4f} | "
        f"F1 macro : {f1_macro:.4f} | "
        f"Durée : {duree / 60:.1f} min | "
        f"VRAM : {vram:.2f} Go"
    )

    # Sauvegarde des meilleurs poids selon le F1 macro
    # ------------------------------------------------------------
    if f1_macro > meilleur_f1:
        meilleur_f1 = f1_macro
        torch.save(model.state_dict(), "meilleurs_poids_camembert.pt")
        print("Meilleurs poids sauvegardés.")

Train 1/3:  13%|█▎        | 1053/8351 [14:11<1:38:23,  1.24it/s]


KeyboardInterrupt: 